# Tip handling on the v1 STAR driver

start date: 2026-09-23

Four tip carriers, one tip size each. Two racks per carrier with tips taken out at random, one
holding its right half, one full. Then pick-ups, returns, drops and discards - including tips of
different kinds in one call.

In [1]:
import datetime

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

simulation = True  # True or False

protocol_mode = "simulation" if simulation else "execution"

run_identifier = "v1_star_tips"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

print(f"logging to {log_path}")

logging to _logs/simulation/v1_star_tips_20260923-103553.log


In [2]:
from pylabrobot.hamilton import STAR

star = STAR(
  simulation=simulation,
)

await star.setup()

# A simulated device answers at once, which leaves a run over before anything watching it has been
# given a turn. Timed, a move takes what the drives would take, at a quarter of it by default.
star.driver.simulate_motion_time = True
star.driver.motion_time_scale = 1.0

2026-09-23 10:35:54,074 - pylabrobot.hamilton.star.driver.master - DEBUG - Setting up STAR on simulation (no link) ...
2026-09-23 10:35:54,075 - pylabrobot.hamilton.star.driver.master - DEBUG - [PHASE 1] Discovery
2026-09-23 10:35:54,075 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RM
2026-09-23 10:35:54,076 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0QM
2026-09-23 10:35:54,077 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RU
2026-09-23 10:35:54,077 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: ru950 13402 0 0 from model the declared arms' X ranges
2026-09-23 10:35:54,077 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0UA
2026-09-23 10:35:54,078 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: ua5952 0 -3232 15172 0 0 from model the declared arms' envelopes
2026-09-23 10:35:54,078 - pylabrobot.hamilton.star.dri

In [3]:
from pylabrobot.resources import set_tip_tracking

set_tip_tracking(True)

In [ ]:
from pylabrobot.visualizer3D.server import Viewer3D

viewer = Viewer3D(star, name="v1_star_tips")
await viewer.start()

viewer on http://127.0.0.1:1338  (websocket 2122)


2026-09-23 10:35:55,376 - pylabrobot.visualizer3D.server - WARNING - refused a websocket without this viewer's token, likely a page left open from an earlier viewer: close it. Repeats are logged at debug.


viewer: a browser connected, drawing with WebGPU


2026-09-23 10:35:57,069 - pylabrobot.visualizer3D.server - DEBUG - refused a websocket without this viewer's token
2026-09-23 10:35:59,070 - pylabrobot.visualizer3D.server - DEBUG - refused a websocket without this viewer's token
2026-09-23 10:36:01,072 - pylabrobot.visualizer3D.server - DEBUG - refused a websocket without this viewer's token
2026-09-23 10:36:03,081 - pylabrobot.visualizer3D.server - DEBUG - refused a websocket without this viewer's token
2026-09-23 10:36:05,106 - pylabrobot.visualizer3D.server - DEBUG - refused a websocket without this viewer's token
2026-09-23 10:36:07,114 - pylabrobot.visualizer3D.server - DEBUG - refused a websocket without this viewer's token
2026-09-23 10:36:09,104 - pylabrobot.visualizer3D.server - DEBUG - refused a websocket without this viewer's token
2026-09-23 10:36:11,307 - pylabrobot.visualizer3D.server - DEBUG - refused a websocket without this viewer's token
2026-09-23 10:36:13,465 - pylabrobot.visualizer3D.server - DEBUG - refused a web

## The racks

In [5]:
from pylabrobot.resources.hamilton import (
  TIP_CAR_480_A00,
  hamilton_96_tiprack_10uL,
  hamilton_96_tiprack_50uL,
  hamilton_96_tiprack_300uL,
  hamilton_96_tiprack_1000uL,
)

racks = {}
for size, make_rack, track in (
  ("10uL", hamilton_96_tiprack_10uL, 1),
  ("50uL", hamilton_96_tiprack_50uL, 7),
  ("300uL", hamilton_96_tiprack_300uL, 13),
  ("1000uL", hamilton_96_tiprack_1000uL, 19),
):
  carrier = TIP_CAR_480_A00(name=f"carrier_{size}")
  for slot in range(4):
    carrier[slot] = racks[size, slot] = make_rack(name=f"{size}_{slot}")
  star.deck.assign_child_resource(carrier, track=track)

In [6]:
import random

wells = [f"{row}{column}" for column in range(1, 13) for row in "ABCDEFGH"]
dice = random.Random(0)

for (size, slot), rack in racks.items():
  if slot < 2:  # tips taken out at random
    kept = set(dice.sample(wells, 60))
    rack.set_tip_state({well: well in kept for well in wells})
  elif slot == 2:  # the right half only
    rack.set_tip_state({well: int(well[1:]) > 6 for well in wells})

{
  name: sum(1 for spot in rack.get_all_items() if spot.has_tip())
  for name, rack in ((rack.name, rack) for rack in racks.values())
}

{'10uL_0': 60,
 '10uL_1': 60,
 '10uL_2': 48,
 '10uL_3': 96,
 '50uL_0': 60,
 '50uL_1': 60,
 '50uL_2': 48,
 '50uL_3': 96,
 '300uL_0': 60,
 '300uL_1': 60,
 '300uL_2': 48,
 '300uL_3': 96,
 '1000uL_0': 60,
 '1000uL_1': 60,
 '1000uL_2': 48,
 '1000uL_3': 96}

In [7]:
full_10, full_50, full_300, full_1000 = (
  racks[size, 3] for size in ("10uL", "50uL", "300uL", "1000uL")
)
holed_300, holed_1000 = racks["300uL", 0], racks["1000uL", 1]
right_half_50 = racks["50uL", 2]

## One kind

In [8]:
await star.pipettes.pick_up_tips([full_300.get_item(f"{row}1") for row in "ABCDEFGH"])

2026-09-23 10:35:55,375 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TTtt01tf0tl0519tv04000tg2tu0
2026-09-23 10:35:55,576 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TPxp03879 03879 03879 03879 03879 03879 03879 03879yp4338 4248 4158 4068 3978 3888 3798 3708tm1 1 1 1 1 1 1 1tt01tp2244tz2164th2828td0
2026-09-23 10:35:55,778 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:35:55,779 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4050, 3783, 3516, 3249, 2982, 2715, 2448, 2181]} from model where the model has the channels along Y
2026-09-23 10:35:55,982 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:35:55,982 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 1

In [9]:
await star.pipettes.request_tool_bottom_z_positions()

2026-09-23 10:35:57,816 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RT
2026-09-23 10:35:57,817 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rt': [1, 1, 1, 1, 1, 1, 1, 1]} from model what the channels carry
2026-09-23 10:35:58,020 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RZ
2026-09-23 10:35:58,021 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': [2828, 2828, 2828, 2828, 2828, 2828, 2828, 2828]} from model where the model has the bottom of what each channel carries
2026-09-23 10:35:58,222 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:35:58,223 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': 31200} from model where the model has channel 0's stop disc
2026-09-23 10:35:58,425 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P2RZ
2026-09-23 10:35:58

{0: 282.8,
 1: 282.8,
 2: 282.8,
 3: 282.8,
 4: 282.8,
 5: 282.8,
 6: 282.8,
 7: 282.8}

In [10]:
await star.pipettes.return_tips()

2026-09-23 10:35:59,928 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TRxp03879 03879 03879 03879 03879 03879 03879 03879yp4338 4248 4158 4068 3978 3888 3798 3708tm1 1 1 1 1 1 1 1tp2244tz2164th2828te2828ti1
2026-09-23 10:36:00,131 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:00,133 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:00,420 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:00,420 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:00,622 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:00,623 - pylabrobot

In [11]:
await star.pipettes.request_stop_disc_z_positions()

2026-09-23 10:36:02,339 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:02,339 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': 26362} from model where the model has channel 0's stop disc
2026-09-23 10:36:02,540 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P2RZ
2026-09-23 10:36:02,541 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': 26362} from model where the model has channel 1's stop disc
2026-09-23 10:36:02,742 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P3RZ
2026-09-23 10:36:02,742 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': 26362} from model where the model has channel 2's stop disc
2026-09-23 10:36:02,943 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P4RZ
2026-09-23 10:36:02,944 - pylabrobot.hamilton.star.driver.simulator - IO - [simul

{0: 282.8,
 1: 282.8,
 2: 282.8,
 3: 282.8,
 4: 282.8,
 5: 282.8,
 6: 282.8,
 7: 282.8}

In [12]:
# a rack with tips taken out at random: what is left is not a column
await star.pipettes.pick_up_tips([spot for spot in holed_300.get_all_items() if spot.has_tip()][:8])

2026-09-23 10:36:03,997 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TPxp03879 03879 03879 03879 03879 00000&yp1458 1098 1008 0918 0828 0000&tm1 1 1 1 1 0&tt01tp2244tz2164th2828td0
2026-09-23 10:36:04,200 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:04,201 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:04,476 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:04,477 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [1458, 1098, 1008, 918, 828, 738, 648, 558]} from model where the model has the channels along Y
2026-09-23 10:36:04,678 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:04,679 - pylabrobot.hamilton.star.driver.simulato

In [13]:
(
  await star.pipettes.request_stop_disc_z_positions(),
  await star.pipettes.request_tool_bottom_z_positions(),
)

2026-09-23 10:36:08,538 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:08,539 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': 31200} from model where the model has channel 0's stop disc
2026-09-23 10:36:08,741 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P2RZ
2026-09-23 10:36:08,742 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': 31200} from model where the model has channel 1's stop disc
2026-09-23 10:36:08,943 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P3RZ
2026-09-23 10:36:08,944 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': 31200} from model where the model has channel 2's stop disc
2026-09-23 10:36:09,146 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P4RZ
2026-09-23 10:36:09,147 - pylabrobot.hamilton.star.driver.simulator - IO - [simul

({0: 334.7,
  1: 334.7,
  2: 334.7,
  3: 334.7,
  4: 334.7,
  5: 334.7,
  6: 334.7,
  7: 334.7},
 {0: 282.8,
  1: 282.8,
  2: 282.8,
  3: 282.8,
  4: 282.8,
  5: 282.8,
  6: 282.8,
  7: 282.8})

In [14]:
await star.pipettes.return_tips()

2026-09-23 10:36:12,227 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TRxp03879 03879 03879 03879 03879 00000&yp1458 1098 1008 0918 0828 0000&tm1 1 1 1 1 0&tp2244tz2164th2828te2828ti1
2026-09-23 10:36:12,430 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:12,431 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [1908, 1818, 1728, 1638, 1548, 1458, 1098, 1008]} from model where the model has the channels along Y
2026-09-23 10:36:12,725 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:12,726 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [1458, 1098, 1008, 918, 828, 738, 648, 558]} from model where the model has the channels along Y
2026-09-23 10:36:12,928 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:12,928 - pylabrobot.hamilton.star.driver.simula

In [15]:
await star.pipettes.pick_up_tips([right_half_50.get_item(f"{row}12") for row in "ABCDEFGH"])

2026-09-23 10:36:17,002 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TTtt02tf0tl0424tv00650tg2tu0
2026-09-23 10:36:17,204 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TPxp03519 03519 03519 03519 03519 03519 03519 03519yp3378 3288 3198 3108 3018 2928 2838 2748tm1 1 1 1 1 1 1 1tt02tp2244tz2164th2923td0
2026-09-23 10:36:17,406 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:17,407 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [1908, 1818, 1728, 1638, 1548, 1458, 1098, 1008]} from model where the model has the channels along Y
2026-09-23 10:36:17,612 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:17,613 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [3378, 3288, 3198, 3108, 3018, 2928, 2838, 2748]} from model where the model has the channels along Y
2026-09-23 1

In [16]:
await star.pipettes.return_tips()

2026-09-23 10:36:19,499 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TRxp03519 03519 03519 03519 03519 03519 03519 03519yp3378 3288 3198 3108 3018 2928 2838 2748tm1 1 1 1 1 1 1 1tp2244tz2164th2923te2923ti1
2026-09-23 10:36:19,701 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:19,702 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [3378, 3288, 3198, 3108, 3018, 2928, 2838, 2748]} from model where the model has the channels along Y
2026-09-23 10:36:19,983 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:19,984 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [3378, 3288, 3198, 3108, 3018, 2928, 2838, 2748]} from model where the model has the channels along Y
2026-09-23 10:36:20,186 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:20,186 - pylabrobot

## Two kinds in one call

One command per kind, in ascending X.

In [17]:
await star.pipettes.pick_up_tips(
  [full_10.get_item(well) for well in ("A1", "B1", "C1", "D1")]
  + [full_1000.get_item(well) for well in ("E1", "F1", "G1", "H1")]
)

2026-09-23 10:36:21,864 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TTtt03tf0tl0219tv00150tg1tu0
2026-09-23 10:36:22,066 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TPxp01179 01179 01179 01179 00000&yp4338 4248 4158 4068 0000&tm1 1 1 1 0&tt03tp2224tz2164th3128td0
2026-09-23 10:36:22,269 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:22,269 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [3378, 3288, 3198, 3108, 3018, 2928, 2838, 2748]} from model where the model has the channels along Y
2026-09-23 10:36:22,474 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:22,474 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3018, 2928, 2838, 2748]} from model where the model has the channels along Y
2026-09-23 10:36:22,676 - pylabrobot.hamilton.st

In [18]:
await star.pipettes.return_tips()

2026-09-23 10:36:26,751 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TRxp01179 01179 01179 01179 00000&yp4338 4248 4158 4068 0000&tm1 1 1 1 0&tp2224tz2144th3128te3128ti1
2026-09-23 10:36:26,954 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:26,955 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:27,297 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:27,298 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:27,499 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:27,500 - pylabrobot.hamilton.star.driver.simulator - IO

## Four kinds

The return is three commands: the 50 uL and 300 uL tips share a collar height.

In [19]:
await star.pipettes.pick_up_tips(
  [
    full_10.get_item("A1"),
    full_50.get_item("B1"),
    full_300.get_item("C1"),
    full_1000.get_item("D1"),
  ],
  use_channels=[0, 1, 2, 3],
)

2026-09-23 10:36:31,552 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TPxp01179 00000&yp4338 0000&tm1 0&tt03tp2224tz2164th3128td0
2026-09-23 10:36:31,754 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:31,754 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:31,957 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:31,957 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:32,159 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:32,159 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz': 

In [20]:
await star.pipettes.return_tips()

2026-09-23 10:36:40,510 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TRxp01179 00000&yp4338 0000&tm1 0&tp2224tz2144th3128te3128ti1
2026-09-23 10:36:40,712 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:40,713 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:40,999 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:41,000 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:41,202 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:41,203 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'rz'

## Channels that are not next to each other, and offsets

In [21]:
await star.pipettes.pick_up_tips(
  [
    full_10.get_item("A2"),
    full_10.get_item("C2"),
    full_1000.get_item("E2"),
    full_1000.get_item("G2"),
  ],
  use_channels=[0, 2, 4, 6],
)

2026-09-23 10:36:47,763 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TPxp01269 00000 01269 00000&yp4338 0000 4158 0000&tm1 0 1 0&tt03tp2224tz2164th3128td0
2026-09-23 10:36:47,965 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:47,965 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:48,169 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:48,169 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:48,371 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:48,371 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation]

In [22]:
await star.pipettes.return_tips()

2026-09-23 10:36:52,235 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TRxp01269 00000 01269 00000&yp4338 0000 4158 0000&tm1 0 1 0&tp2224tz2144th3128te3128ti1
2026-09-23 10:36:52,438 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:52,438 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:52,740 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:52,740 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:52,942 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:52,942 - pylabrobot.hamilton.star.driver.simulator - IO - [simulatio

In [23]:
from pylabrobot.resources import Coordinate

await star.pipettes.pick_up_tips(
  [
    full_10.get_item("A3"),
    full_10.get_item("B3"),
    full_1000.get_item("C3"),
    full_1000.get_item("D3"),
  ],
  offsets=[
    Coordinate(0.5, 0.5, 0),
    Coordinate(-0.5, 0.5, 0),
    Coordinate(0.5, -0.5, 0),
    Coordinate(0, 0, 0),
  ],
)

2026-09-23 10:36:56,983 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TPxp01364 01354 00000&yp4343 4253 0000&tm1 1 0&tt03tp2224tz2164th3128td0
2026-09-23 10:36:57,187 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:57,187 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:57,392 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:36:57,393 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4343, 4253, 4158, 4068, 3978, 3888, 3798, 3708]} from model where the model has the channels along Y
2026-09-23 10:36:57,594 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:36:57,595 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simula

In [24]:
await star.pipettes.return_tips()

2026-09-23 10:37:03,734 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0TRxp01359 01359 00000&yp4338 4248 0000&tm1 1 0&tp2224tz2144th3128te3128ti1
2026-09-23 10:37:03,936 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:37:03,937 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4343, 4253, 4158, 4068, 3973, 3883, 3793, 3703]} from model where the model has the channels along Y
2026-09-23 10:37:04,236 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RY
2026-09-23 10:37:04,236 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simulation: {'ry': [4338, 4248, 4158, 4068, 3973, 3883, 3793, 3703]} from model where the model has the channels along Y
2026-09-23 10:37:04,438 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: P1RZ
2026-09-23 10:37:04,438 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] read: simu

## Dropped elsewhere, let go on the deck, discarded

In [ ]:
await star.pipettes.pick_up_tips([full_300.get_item(well) for well in ("A4", "B4", "C4", "D4")])

In [ ]:
# into spots they did not come from
await star.pipettes.drop_tips(
  [spot for spot in holed_300.get_all_items() if not spot.has_tip()][:4]
)

In [ ]:
await star.pipettes.pick_up_tips(
  [spot for spot in holed_1000.get_all_items() if spot.has_tip()][:4]
)

In [ ]:
# let go on the deck, a place per channel
place = full_1000.get_item("A1").get_location_wrt(star.deck, x="c", y="c", z="b")
await star.pipettes.drop_tips(
  [place + Coordinate(0, -30 - 18 * channel, 0) for channel in range(4)]
)

In [ ]:
await star.pipettes.pick_up_tips([full_50.get_item(f"{row}5") for row in "ABCDEFGH"])

In [ ]:
await star.pipettes.discard_tips()

In [ ]:
await star.pipettes.pick_up_tips([full_300.get_item(well) for well in ("A6", "B6", "C6", "D6")])

In [ ]:
await star.pipettes.discard_tips(use_channels=[0, 1])

In [ ]:
await star.pipettes.return_tips(use_channels=[2, 3])

## Where everything ended

In [ ]:
print(await star.pipettes.request_stop_disc_z_positions())
print([star.pipettes.get_mounted_tip(channel) is not None for channel in range(8)])